# 🏠 Real Estate Price Prediction System
## Interactive Prediction Interface

This notebook provides a complete interface to:
1. Load the trained XGBoost model
2. Input property details interactively
3. Get accurate price predictions
4. View confidence metrics and comparisons

## 1. Import Libraries & Load Model

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Add model directory to path
sys.path.insert(0, '/src/models/Xgboost')

# Style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ Libraries imported successfully!')

In [ ]:
# Load the trained model and preprocessing pipeline
model_dir = Path('src/models/Xgboost')
model_files = list(model_dir.glob('real_estate_model_*.joblib'))

if not model_files:
    print('❌ No trained model found!')
    print('📌 First run the training: python src/models/Xgboost/train.py')
else:
    # Get the most recent model
    model_path = max(model_files, key=os.path.getctime)
    print(f'📦 Loading model: {model_path.name}')
    
    components = joblib.load(model_path)
    preprocessor = components['preprocessor']
    model = components['model']
    feature_names = components['feature_names']
    model_type = components['type']
    
    print(f'✅ Model loaded for: {model_type}')
    print(f'📊 Number of features: {len(feature_names)}')

## 2. Define Input Features & Validation

In [ ]:
# Define feature types and ranges
feature_config = {
    'location': {'type': 'text', 'example': 'Guéliz, Marrakech', 'required': True},
    'surface': {'type': 'float', 'min': 10, 'max': 2000, 'unit': 'm²', 'required': True},
    'rooms': {'type': 'int', 'min': 0, 'max': 20, 'required': True},
    'bedrooms': {'type': 'int', 'min': 0, 'max': 20, 'required': True},
    'bathrooms': {'type': 'int', 'min': 0, 'max': 10, 'required': True},
    'property_category': {'type': 'categorical', 'options': ['Apartment', 'Villa', 'House', 'Other'], 'required': True},
    'type': {'type': 'categorical', 'options': ['For Sale', 'For Rent'], 'required': True},
    'terrace': {'type': 'bool', 'required': False},
    'garage': {'type': 'bool', 'required': False},
    'elevator': {'type': 'bool', 'required': False},
    'concierge': {'type': 'bool', 'required': False},
    'pool': {'type': 'bool', 'required': False},
    'security': {'type': 'bool', 'required': False},
    'garden': {'type': 'bool', 'required': False},
}

print('✅ Feature configuration loaded')
print(f'\n📋 Expected features: {list(feature_config.keys())}')

## 3. Interactive Input Form

In [ ]:
print('\n' + '='*70)
print('🏠 ENTER PROPERTY DETAILS'.center(70))
print('='*70)

# Initialize property data dictionary
property_data = {}

# Get inputs from user
print('\n🗺️  LOCATION')
print('Format: District, City (e.g., "Guéliz, Marrakech")')
property_data['location'] = input('Location: ').strip()

print('\n📐 DIMENSIONS')
property_data['surface'] = float(input('Surface (m²): '))
property_data['rooms'] = int(input('Total Rooms: '))
property_data['bedrooms'] = int(input('Bedrooms: '))
property_data['bathrooms'] = int(input('Bathrooms: '))

print('\n🏢 PROPERTY TYPE')
print('Options: Apartment, Villa, House, Other')
property_data['property_category'] = input('Category: ').strip()

print('\n💼 LISTING TYPE')
print('Options: For Sale, For Rent')
property_data['type'] = input('Type: ').strip()

print('\n✨ AMENITIES (yes/no)')
amenities = ['terrace', 'garage', 'elevator', 'concierge', 'pool', 'security', 'garden']
for amenity in amenities:
    response = input(f'{amenity.capitalize()}? (y/n): ').strip().lower()
    property_data[amenity] = response in ['y', 'yes', '1', 'true']

print('\n✅ Input received!')

## 4. Feature Engineering & Preprocessing

In [ ]:
# Create DataFrame from input
df_input = pd.DataFrame([property_data])

# Apply same feature engineering as during training
print('\n🔧 Applying feature engineering...')

# Interaction features
df_input['rooms_per_surface'] = df_input['rooms'] / df_input['surface'].replace(0, 1)
df_input['bathrooms_per_surface'] = df_input['bathrooms'] / df_input['surface'].replace(0, 1)
df_input['bedrooms_per_surface'] = df_input['bedrooms'] / df_input['surface'].replace(0, 1)

df_input['surface_x_rooms'] = df_input['surface'] * df_input['rooms']
df_input['surface_x_bathrooms'] = df_input['surface'] * df_input['bathrooms']
df_input['surface_x_bedrooms'] = df_input['surface'] * df_input['bedrooms']

# Polynomial features
df_input['surface_sq'] = df_input['surface']**2
df_input['surface_cub'] = df_input['surface']**3

# Location features
df_input['location_city'] = df_input['location'].str.split(',').str[0].str.strip()
df_input['location_district'] = df_input['location'].str.split(',').str[1].str.strip().fillna('Other')

# Luxury features count
df_input['luxury_features_count'] = df_input[['concierge', 'pool', 'security', 'garden']].fillna(0).sum(axis=1)

# Select only required features
X_input = df_input[feature_names]

# Preprocess
X_processed = preprocessor.transform(X_input)

print('✅ Feature engineering completed!')
print(f'📊 Processed features shape: {X_processed.shape}')

## 5. Generate Price Prediction

In [ ]:
# Make prediction
print('\n🔮 Generating prediction...')

# Predict (in log scale)
price_log_pred = model.predict(X_processed)[0]

# Convert back from log scale
predicted_price = np.expm1(price_log_pred)

# Calculate confidence metrics
# Use model's predictions variance for confidence intervals
std_error = np.std(model.predict(X_processed))
confidence_95 = 1.96 * std_error

price_lower = predicted_price * 0.90  # Conservative 10% margin
price_upper = predicted_price * 1.10  # Conservative 10% margin

print('✅ Prediction generated!')
print(f'\n💰 Base Predicted Price: {predicted_price:,.0f} MAD')
print(f'📊 Price Range (±10%): {price_lower:,.0f} - {price_upper:,.0f} MAD')

## 6. Display Results & Visualizations

In [ ]:
# Display comprehensive results
print('\n' + '='*70)
print('📊 PREDICTION RESULTS'.center(70))
print('='*70)

# Input summary
print('\n🏠 PROPERTY DETAILS:')
print(f'   Location: {property_data["location"]}')
print(f'   Type: {property_data["type"]} - {property_data["property_category"]}')
print(f'   Surface: {property_data["surface"]} m²')
print(f'   Rooms: {property_data["rooms"]} (Bedrooms: {property_data["bedrooms"]}, Bathrooms: {property_data["bathrooms"]})')

# Amenities
amenities_yes = [k for k, v in property_data.items() 
                 if k in amenities and v]
if amenities_yes:
    print(f'   Amenities: {", ".join([a.capitalize() for a in amenities_yes])}')

# Prediction
print('\n' + '-'*70)
print('💰 PREDICTED PRICE')
print('-'*70)
print(f'   Base Price:      {predicted_price:>20,.0f} MAD')
print(f'   Price per m²:    {predicted_price/property_data["surface"]:>20,.0f} MAD/m²')
print(f'   Estimated Range: {price_lower:>20,.0f} - {price_upper:,.0f} MAD')
print('\n' + '='*70)

# Timestamp
print(f'\n⏰ Prediction made at: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'🤖 Model Type: {model_type}')

In [ ]:
# Visualization 1: Predicted Price Components
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price breakdown
ax1 = axes[0]
price_components = {
    'Base Price': predicted_price,
    'Range Lower': price_lower - predicted_price,
    'Range Upper': price_upper - predicted_price
}

colors = ['#2ecc71', '#e74c3c', '#f39c12']
ax1.barh(list(price_components.keys()), [predicted_price, price_upper - predicted_price, price_upper - price_lower],
         color=colors)
ax1.set_xlabel('Price (MAD)')
ax1.set_title('🏠 Predicted Price Distribution')
ax1.grid(axis='x', alpha=0.3)

# Property metrics
ax2 = axes[1]
metrics = {
    'Surface': property_data['surface'],
    'Rooms': property_data['rooms'],
    'Bedrooms': property_data['bedrooms'],
    'Bathrooms': property_data['bathrooms']
}

ax2.bar(metrics.keys(), metrics.values(), color='#3498db', alpha=0.7)
ax2.set_ylabel('Value')
ax2.set_title('📐 Property Specifications')
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('✅ Visualization displayed')

In [ ]:
# Summary Report
print('\n' + '='*70)
print('📋 PREDICTION SUMMARY REPORT'.center(70))
print('='*70)

summary_data = {
    'Location': property_data['location'],
    'Property Type': property_data['property_category'],
    'Listing Type': property_data['type'],
    'Surface (m²)': f"{property_data['surface']:.1f}",
    'Rooms': property_data['rooms'],
    'Bedrooms': property_data['bedrooms'],
    'Bathrooms': property_data['bathrooms'],
    'Amenities': len([k for k, v in property_data.items() if k in amenities and v]),
    '---': '---',
    'Predicted Price': f"{predicted_price:,.0f} MAD",
    'Price per m²': f"{predicted_price/property_data['surface']:,.0f} MAD/m²",
    'Confidence Range': f"{price_lower:,.0f} - {price_upper:,.0f} MAD",
    'Confidence Level': '±10%'
}

for key, value in summary_data.items():
    if key == '---':
        print('-'*70)
    else:
        print(f'{key:<25} {str(value):>40}')

print('='*70)

## 📌 Notes & Usage Tips

**✅ This prediction system:**
- Uses a trained XGBoost model with real estate data from Morocco
- Provides predictions in Moroccan Dirham (MAD)
- Includes ±10% confidence range
- Accounts for property features like amenities, location, and specifications

**💡 Tips for better predictions:**
1. Enter accurate property location (District, City)
2. Specify all amenities accurately
3. For new locations, the model interpolates based on similar areas
4. Price predictions are based on historical market data

**⚠️ Limitations:**
- Predictions are estimates based on training data
- Market conditions change over time
- Outlier properties may have different values
- Use as a reference, not absolute value

**📞 For more information:**
- See `README.md` for complete documentation
- Check `docs/model_card.md` for model details
- Visit `src/models/Xgboost/` for training scripts